<a href="https://colab.research.google.com/github/priyansh-commits/Deep_Learning/blob/main/multi_output_dl_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import kagglehub
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# ==========================================
# 1. Download UTKFace
# ==========================================

path = kagglehub.dataset_download("jangedoo/utkface-new")
print("Dataset path:", path)


# ==========================================
# 2. Find all images
# ==========================================

image_files = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            image_files.append(os.path.join(root, file))

print("Total images:", len(image_files))


# ==========================================
# 3. Load images + Age + Gender
# ==========================================

IMG_SIZE = 64

X = []
age = []
gender = []

for file in image_files:

    try:
        # Example:
        # 100_0_0_20170119202909553.jpg
        filename = os.path.basename(file)

        parts = filename.split("_")

        # Age
        a = int(parts[0])

        # Gender
        g = int(parts[1])

        # Only valid gender labels
        if g not in [0, 1]:
            continue

        # Read image
        img = cv2.imread(file)

        if img is None:
            continue

        # BGR -> RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Resize
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        X.append(img)
        age.append(a)
        gender.append(g)

    except:
        continue


# ==========================================
# 4. Convert to NumPy
# ==========================================

X = np.array(X, dtype=np.float32) / 255.0

age = np.array(age, dtype=np.float32)

gender = np.array(gender, dtype=np.float32)


print("\nDataset:")
print("X:", X.shape)
print("Age:", age.shape)
print("Gender:", gender.shape)


# ==========================================
# 5. Train/Test Split
# ==========================================

X_train, X_test, age_train, age_test, gender_train, gender_test = train_test_split(
    X,
    age,
    gender,
    test_size=0.20,
    random_state=42,
    stratify=gender
)


# ==========================================
# 6. Final shapes
# ==========================================

print("\nTRAIN")
print("X_train:", X_train.shape)
print("age_train:", age_train.shape)
print("gender_train:", gender_train.shape)

print("\nTEST")
print("X_test:", X_test.shape)
print("age_test:", age_test.shape)
print("gender_test:", gender_test.shape)

Using Colab cache for faster access to the 'utkface-new' dataset.
Dataset path: /kaggle/input/utkface-new
Total images: 66976

Dataset:
X: (66974, 64, 64, 3)
Age: (66974,)
Gender: (66974,)

TRAIN
X_train: (53579, 64, 64, 3)
age_train: (53579,)
gender_train: (53579,)

TEST
X_test: (13395, 64, 64, 3)
age_test: (13395,)
gender_test: (13395,)


In [1]:
import tensorflow as tf

tf.debugging.set_log_device_placement(False)

In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
print('h')

h


In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow import keras
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras import Model


In [7]:
# from keras.preprocessing.image import ImageDataGenerator

# datagen = ImageDataGenerator(
#         rotation_range=40,
#         width_shift_range=0.2,
#         height_shift_range=0.2,
#         rescale=1./255,
#         shear_range=0.2,
#         zoom_range=0.2,
#         horizontal_flip=True,
#         fill_mode='nearest')

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_temp, age_train, age_temp, gender_train, gender_temp = train_test_split(
    X, age, gender,
    test_size=0.30,
    random_state=42,
    stratify=gender
)

X_val, X_test, age_val, age_test, gender_val, gender_test = train_test_split(
    X_temp, age_temp, gender_temp,
    test_size=0.50,
    random_state=42,
    stratify=gender_temp
)

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, Model

IMG_SIZE = 64

# Pretrained CNN
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Initially freeze it
base_model.trainable = False


# Input
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# CNN feature extraction
x = base_model(inputs, training=False)

# Global pooling
x = layers.GlobalAveragePooling2D()(x)

# Nonlinear shared layers
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)


# AGE OUTPUT
age_output = layers.Dense(
    1,
    activation='linear',
    name='age'
)(x)


# GENDER OUTPUT
gender_output = layers.Dense(
    1,
    activation='sigmoid',
    name='gender'
)(x)


# Final model
model = Model(
    inputs=inputs,
    outputs=[age_output, gender_output]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetb0      │ (None, 2, 2,      │  4,049,571 │ input_layer_1[0]… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ efficientnetb0[0… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    327,936 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ age (Dense)         │ (None, 1)         │        129 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender (Dense)      │ (None, 1)         │        129 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,410,661 (16.83 MB)

 Trainable params: 361,090 (1.38 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),

    loss={
        'age': 'mse',
        'gender': 'binary_crossentropy'
    },

    metrics={
        'age': ['mae'],
        'gender': ['accuracy']
    }
)

In [1]:
history = model.fit(
    X_train,
    {
        'age': age_train,
        'gender': gender_train
    },

    validation_data=(
        X_val,
        {
            'age': age_val,
            'gender': gender_val
        }
    ),

    epochs=10,
    batch_size=64
)

NameError: name 'model' is not defined

In [ ]:
age_pred, gender_pred = model.predict(X_test)

age_pred = age_pred.flatten()
gender_pred = (gender_pred.flatten() >= 0.5).astype(int)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(age_test, age_pred)
rmse = mean_squared_error(
    age_test,
    age_pred
) ** 0.5

print("Age MAE:", mae)
print("Age RMSE:", rmse)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        gender_test,
        gender_pred
    )
)

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
import tensorflow as tf

print("GPU:", tf.config.list_physical_devices("GPU"))

tf.debugging.set_log_device_placement(True)

with tf.device("/GPU:0"):
    a = tf.random.normal((2000, 2000))
    b = tf.matmul(a, a)

print("Result device:", b.device)

In [ ]:
!nvidia-smi